## SETUP & KONFIGURASI

In [ ]:
import clickhouse_connect
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import scipy

# Tampilkan versi semua library (WAJIB)
print("=== VERSI LIBRARY ===")
print(f"clickhouse-connect : {clickhouse_connect.__version__}")
print(f"pandas             : {pd.__version__}")
print(f"numpy              : {np.__version__}")
print(f"scikit-learn       : {sklearn.__version__}")
print(f"matplotlib         : {matplotlib.__version__}")
print(f"seaborn            : {sns.__version__}")
print(f"scipy              : {scipy.__version__}")

# Variabel konfigurasi koneksi
HOST      = 'overproficiently-unbenevolent-gregoria.ngrok-free.dev'
PORT      = 443
USERNAME  = 'mahasiswa'
PASSWORD  = 'bigdata123'
DATABASE  = 'praktikum'
TABLE_SRC = 'final_pipeline'
TABLE_DST = 'clustering_results'
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
print(f"\nRandom seed ditetapkan: {RANDOM_SEED}")

# Koneksi ke ClickHouse
print("\nMenghubungkan ke ClickHouse...")
client = clickhouse_connect.get_client(
    host=HOST,
    port=PORT,
    username=USERNAME,
    password=PASSWORD,
    database=DATABASE,
    secure=True
)
print(f"Koneksi berhasil! Versi server: {client.server_version}")

## EXPORT DATA HASIL PREPROCESSING KE PANDAS

In [ ]:
# Query agregasi per perusahaan — fitur statistik pola publikasi
# Keputusan: clustering dilakukan pada level perusahaan (361 entitas unik)
# Alasan: kolom teks tidak bisa langsung di-cluster tanpa NLP;
#         fitur waktu (hour, minute, day_of_week, is_weekend) merepresentasikan
#         pola perilaku publikasi berita setiap perusahaan.

print("Mengambil dan mengagregasi data dari ClickHouse...")

query = """
    SELECT
        cik_clean,
        cleaned_company,

        -- Volume publikasi
        count(*) AS total_articles,

        -- Pola jam publikasi
        avg(hour)               AS avg_hour,
        stddevPop(hour)         AS std_hour,
        min(hour)               AS min_hour,
        max(hour)               AS max_hour,

        -- Pola menit publikasi
        avg(minute)             AS avg_minute,
        stddevPop(minute)       AS std_minute,

        -- Pola hari dalam minggu
        avg(day_of_week)        AS avg_day_of_week,
        stddevPop(day_of_week)  AS std_day_of_week,

        -- Proporsi publikasi di akhir pekan
        avg(is_weekend)         AS ratio_weekend,

        -- Publikasi di luar jam kerja (sebelum 08.00 atau setelah 18.00)
        avgIf(1, hour < 8 OR hour >= 18) AS ratio_off_hours,

        -- Publikasi di jam sibuk (08.00–12.00)
        avgIf(1, hour >= 8 AND hour < 12) AS ratio_morning_peak,

        -- Publikasi sore (12.00–18.00)
        avgIf(1, hour >= 12 AND hour < 18) AS ratio_afternoon

    FROM praktikum.final_pipeline
    GROUP BY cik_clean, cleaned_company
    ORDER BY total_articles DESC
"""

df = client.query_df(query)

print(f"✅ Data berhasil diambil!")
print(f"   Jumlah perusahaan (baris) : {df.shape[0]}")
print(f"   Jumlah fitur (kolom)      : {df.shape[1]}")
print(f"   Ukuran DataFrame          : {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"\nKolom: {list(df.columns)}")
print(f"\nSample data:")
print(df.head())
print(f"\nStatistik deskriptif:")
print(df.describe())

## PRE-PROCESSING DI PYTHON

### 1. Pilih fitur untuk clustering

In [ ]:
# Keputusan: kolom cik_clean & cleaned_company digunakan sebagai
# identitas, BUKAN sebagai fitur model (teks tidak di-encode
# karena tidak merepresentasikan pola perilaku secara numerik)

feature_cols = [
    'total_articles',
    'avg_hour',
    'std_hour',
    'min_hour',
    'max_hour',
    'avg_minute',
    'std_minute',
    'avg_day_of_week',
    'std_day_of_week',
    'ratio_weekend',
    'ratio_off_hours',
    'ratio_morning_peak',
    'ratio_afternoon'
]

# Simpan identitas perusahaan terpisah
df_identity = df[['cik_clean', 'cleaned_company']].copy()

# Matrix fitur
X = df[feature_cols].copy()

print("=== 4.1 FITUR YANG DIGUNAKAN ===")
print(f"Jumlah fitur  : {len(feature_cols)}")
print(f"Jumlah sampel : {X.shape[0]}")
print(f"\nCek missing values setelah agregasi:")
print(X.isnull().sum())

# Isi NaN pada std (perusahaan dengan 1 artikel → std = NaN → 0)
X = X.fillna(0)
print(f"\nMissing values setelah fillna(0): {X.isnull().sum().sum()}")

### 2 - Scaling dengan StandardScaler

In [ ]:
# Keputusan: StandardScaler dipakai karena clustering berbasis
# jarak (K-Means, DBSCAN) sensitif terhadap skala fitur.
# total_articles bisa sangat besar dibanding ratio_* yang 0–1.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

print("\n=== 4.2 HASIL SCALING (StandardScaler) ===")
print("Statistik setelah scaling (mean ≈ 0, std ≈ 1):")
print(X_scaled_df.describe().round(4))

### 3 - PCA untuk reduksi dimensi (visualisasi 2D)

In [ ]:

# Keputusan: PCA 2 komponen digunakan KHUSUS untuk visualisasi
# scatter plot oleh Orang 4. Model clustering tetap memakai
# X_scaled (13 fitur) agar informasi tidak hilang.

pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)

print("\n=== 4.3 HASIL PCA ===")
print(f"Explained variance ratio : {pca.explained_variance_ratio_}")
print(f"Total variance explained : {sum(pca.explained_variance_ratio_):.2%}")
print(f"Shape X_pca              : {X_pca.shape}")

# Simpan PCA ke DataFrame dengan identitas
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca = pd.concat([df_identity.reset_index(drop=True), df_pca], axis=1)

print(f"\nSample df_pca:")
print(df_pca.head())

### 4 - Ringkasan variabel untuk Orang 2

In [ ]:
print("\n=== RINGKASAN OUTPUT UNTUK ORANG 2 ===")
print(f"X_scaled  → shape {X_scaled.shape}  | input model clustering")
print(f"X_pca     → shape {X_pca.shape}     | visualisasi scatter 2D")
print(f"df        → shape {df.shape}        | DataFrame lengkap")
print(f"df_pca    → shape {df_pca.shape}    | PCA + identitas perusahaan")
print(f"RANDOM_SEED = {RANDOM_SEED}")
print("\n✅ Pre-processing selesai! Data siap untuk modeling (Orang 2).")